# Synthetic Data for Rare Events

## What This Is
Class imbalance is pervasive in real-world ML: fraud detection (0.1% positive), medical diagnosis (rare diseases), network intrusion detection. Synthetic data generation for rare events addresses this through oversampling the minority class.

## Approaches

**SMOTE (Synthetic Minority Oversampling Technique)**: Interpolates between k-nearest neighbors in feature space. For each minority sample, pick a random neighbor and generate a synthetic point along the line between them. Simple, effective, but operates in feature space — may generate unrealistic combinations for categorical features.

**SMOTE variants**:
- *Borderline-SMOTE*: Only oversample near the decision boundary
- *ADASYN*: Adaptive generation — more samples where classification is harder
- *SMOTENC*: SMOTE for numerical + categorical mixed data

**GAN-based oversampling**: Train a conditional GAN on the minority class. Produces more realistic samples but requires enough minority samples to train.

In [ ]:
import numpy as np
from typing import Tuple, List

np.random.seed(42)

# Create imbalanced fraud detection dataset
N_majority = 1000  # legitimate transactions
N_minority = 50    # fraud (5% imbalance ratio)

# Majority class: normal transactions
X_maj = np.random.randn(N_majority, 5)
X_maj[:, 0] += 1   # shift mean slightly

# Minority class: fraud — different distribution, overlap region
X_min = np.random.randn(N_minority, 5) * 1.5
X_min[:, 2] += 2   # different feature pattern

y_maj = np.zeros(N_majority)
y_min = np.ones(N_minority)

X = np.vstack([X_maj, X_min])
y = np.concatenate([y_maj, y_min])

print(f'Dataset imbalance ratio: {N_majority}:{N_minority}')
print(f'Minority class fraction: {N_minority/(N_majority+N_minority):.3f}')


In [ ]:
# SMOTE implementation

def smote(X_minority: np.ndarray, n_synthetic: int, k: int = 5) -> np.ndarray:
    """
    SMOTE: Synthetic Minority Oversampling TEchnique.
    For each minority sample, find k nearest neighbors,
    generate synthetic sample along the connecting line.
    """
    n_min = len(X_minority)
    synthetic = []
    
    # Compute pairwise distances
    dists = np.array([
        np.linalg.norm(X_minority - X_minority[i], axis=1)
        for i in range(n_min)
    ])
    
    for _ in range(n_synthetic):
        # Pick random minority sample
        i = np.random.randint(n_min)
        # Find k nearest neighbors (excluding self)
        sorted_idx = np.argsort(dists[i])
        neighbors = sorted_idx[1:k+1]  # skip self
        # Pick random neighbor
        j = np.random.choice(neighbors)
        # Generate point along the line between i and j
        lam = np.random.uniform(0, 1)
        synth_point = X_minority[i] + lam * (X_minority[j] - X_minority[i])
        synthetic.append(synth_point)
    
    return np.array(synthetic)

# Generate SMOTE synthetic samples
n_to_generate = N_majority - N_minority  # balance to 1:1
X_smote = smote(X_min, n_to_generate)

# Balanced dataset
X_balanced = np.vstack([X_maj, X_min, X_smote])
y_balanced = np.concatenate([y_maj, y_min, np.ones(n_to_generate)])

print(f'SMOTE generated: {len(X_smote)} synthetic minority samples')
print(f'Balanced dataset: {(y_balanced==0).sum()} majority, {(y_balanced==1).sum()} minority')


In [ ]:
# Borderline-SMOTE: only oversample near the decision boundary

def borderline_smote(X_minority: np.ndarray, X_all: np.ndarray, 
                     n_synthetic: int, k: int = 5, k_all: int = 10) -> np.ndarray:
    """
    Borderline-SMOTE: identify minority samples near the boundary
    (those whose k-NN from all classes includes many majority samples).
    Only oversample from those borderline samples.
    """
    n_min = len(X_minority)
    
    # For each minority sample, find k_all nearest in full dataset
    borderline_idx = []
    for i in range(n_min):
        dists = np.linalg.norm(X_all - X_minority[i], axis=1)
        knn_idx = np.argsort(dists)[1:k_all+1]
        # Count majority neighbors (y=0)
        n_majority_neighbors = sum(1 for idx in knn_idx if idx < len(X_all) - n_min)
        # Borderline: more than half neighbors are majority
        ratio = n_majority_neighbors / k_all
        if 0.5 <= ratio < 1.0:  # borderline, not noise
            borderline_idx.append(i)
    
    if len(borderline_idx) == 0:
        borderline_idx = list(range(n_min))  # fallback
    
    X_border = X_minority[borderline_idx]
    synthetic = smote(X_border, n_synthetic, k=min(k, len(X_border)-1))
    return synthetic, borderline_idx

X_bsmote, border_idx = borderline_smote(X_min, X, n_to_generate)
print(f'Borderline-SMOTE: {len(border_idx)} borderline samples identified')
print(f'Generated {len(X_bsmote)} synthetic minority samples from borderline region')

# Compare classifier performance
def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
def train_lr(Xtr, ytr, lr=0.05, epochs=300):
    w = np.zeros(Xtr.shape[1]); b = 0.0
    for _ in range(epochs):
        e = sigmoid(Xtr @ w + b) - ytr
        w -= lr * (Xtr.T @ e) / len(ytr); b -= lr * e.mean()
    return w, b

# Test on held-out set
X_test_pos = np.random.randn(100, 5) * 1.5; X_test_pos[:, 2] += 2
X_test_neg = np.random.randn(100, 5); X_test_neg[:, 0] += 1
X_test = np.vstack([X_test_neg, X_test_pos])
y_test = np.array([0]*100 + [1]*100)

mu, s = X.mean(0), X.std(0)
norm = lambda Xn: (Xn - mu) / (s + 1e-8)

# Imbalanced
w0, b0 = train_lr(norm(X), y)
p0 = (sigmoid(norm(X_test) @ w0 + b0) > 0.5).astype(int)

# SMOTE balanced
Xb = np.vstack([X_maj, X_min, X_smote])
yb = np.concatenate([y_maj, y_min, np.ones(len(X_smote))])
w1, b1 = train_lr(norm(Xb), yb)
p1 = (sigmoid(norm(X_test) @ w1 + b1) > 0.5).astype(int)

def f1(pred, true, cls=1):
    tp = ((pred == cls) & (true == cls)).sum()
    fp = ((pred == cls) & (true != cls)).sum()
    fn = ((pred != cls) & (true == cls)).sum()
    if tp + fp == 0 or tp + fn == 0: return 0.0
    p = tp / (tp + fp); r = tp / (tp + fn)
    return 2 * p * r / (p + r) if p + r > 0 else 0.0

print(f'\nF1-score on minority class (fraud):')
print(f'  Imbalanced training:   {f1(p0, y_test):.3f}')
print(f'  SMOTE-balanced:        {f1(p1, y_test):.3f}')
